In [30]:
import pandas as pd
import matplotlib as plt
from metrics import metrics
from model import generics
pd.set_option('display.max_rows', 2000)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [102]:
experiment_id = 'pertexps'

In [103]:
df_mean_metrics, df_all_metrics, df_prevs = metrics.open_fold_result(experiment_id)

In [104]:
df_mean_metrics#[df_mean_metrics.reset_index()['model'].str.contains("mlp-").values]

MSE          RMSE         MAPE  \
ts            model                                                     
Unemployment  arima           2.978551e+03     54.576100     7.012015   
              mlpgs           1.317445e+03     36.253273     4.530343   
              mlpkpssfalse    1.257599e+03     35.441220     4.490128   
              mlpkpssfalsep1  1.263940e+03     35.533396     4.273653   
              mlpkpssfalsep2  1.194990e+03     34.556246     4.218171   
              mlpkpssfalsep3  1.378923e+03     37.112143     4.474092   
              mlpwk0.61005    1.328637e+03     36.422281     4.314731   
              mlpwk11005      1.250247e+03     35.326696     4.202943   
              svrgs           1.935252e+03     43.991502     5.363710   
              svrwk0.6505     2.234920e+03     47.274940     5.726779   
ausbee        arima           1.851701e+02     13.607720     2.445095   
              mlpgs           2.634443e+02     16.224474     2.871810   
              mlpkpssfalse    2.495322e+02     15.773261     2.765758   
              mlpkpssfalsep1  3.166997e+02     17.790413     3.203947   
              mlpkpssfalsep2  3.310681e+02     18.188745     3.312143   
              mlpkpssfalsep3  3.571557e+02     18.894978     3.525176   
              mlpwk0.61005    1.002970e+04     90.301617    16.096839   
              mlpwk11005      1.370786e+03     34.660087     6.251220   
              svrgs           3.635586e+02     19.067212     3.483389   
              svrwk0.6505     1.505774e+06   1227.099833   285.810532   
austres       arima           2.925290e+02     17.103479     0.075877   
              mlpgs           3.158341e+02     17.771329     0.074567   
              mlpkpssfalse    2.121747e+03     42.316535     0.230565   
              mlpkpssfalsep1  3.030502e+02     17.404905     0.084490   
              mlpkpssfalsep2  3.213592e+02     17.887057     0.086218   
              mlpkpssfalsep3  3.658122e+02     19.054031     0.092524   
              mlpwk0.61005    3.188609e+02     17.844809     0.086432   
              mlpwk11005      3.190547e+02     17.839236     0.086425   
              svrgs           3.348059e+02     18.297702     0.080785   
              svrwk0.6505     3.251646e+08  18032.321671   104.370228   
coloradoRiver arima           2.769386e-01      0.526250    93.935436   
              mlpgs           2.483630e-01      0.498215    77.226950   
              mlpkpssfalse    2.497202e-01      0.499563   100.738194   
              mlpkpssfalsep1  2.252985e-01      0.474559    92.121111   
              mlpkpssfalsep2  2.175353e-01      0.466346    85.285856   
              mlpkpssfalsep3  2.210226e-01      0.469983    87.321495   
              mlpwk0.61005    2.262175e-01      0.475285    83.075292   
              mlpwk0.81005    2.289332e-01      0.478271    90.387409   
              mlpwk11005      2.204369e-01      0.468930    87.788905   
              svrgs           2.639992e-01      0.513808    68.067915   
              svrwk0.6505     3.377271e+02     18.377352  2998.796556   
gasoline      arima           5.964224e+04    244.217611     2.216460   
              mlpgs           6.120519e+04    247.394586     2.194634   
              mlpkpssfalse    6.414784e+04    253.272202     2.319554   
              mlpkpssfalsep1  6.548213e+04    255.878632     2.345376   
              mlpkpssfalsep2  6.645167e+04    257.773694     2.361211   
              mlpkpssfalsep3  6.672063e+04    258.286232     2.366742   
              mlpwk0.61005    6.913146e+04    262.787022     2.410329   
              mlpwk11005      7.014855e+04    264.696197     2.430625   
              svrgs           6.340822e+04    251.809890     2.203433   
heartrate     arima           4.069414e-01      0.637920     0.594484   
              mlpgs           1.089161e+00      1.027196     1.016918   
              mlpkpssfalse    8.209328e-01      0.898068     0.862110   
              mlpkpssfal

In [87]:
metric = 'RMSE'
df_arima = pd.DataFrame()
df_svr = pd.DataFrame()
df_mlp = pd.DataFrame()

proposed_list = []
for k, df in df_mean_metrics.reset_index().groupby('ts'):
    arima = df[df['model'] == 'arima'][metric].iloc[0]
    svr = df[df['model'] == 'arima'][metric].iloc[0]
    mlp = df[df['model'] == 'arima'][metric].iloc[0]

    df_arima[k] = (df.set_index('model')[metric] < arima)
    df_svr[k] = (df.set_index('model')[metric] < svr)
    df_mlp[k] = (df.set_index('model')[metric] < mlp)

    svr_proposed = df[( (df['model'].str.contains("svr-")) & (df['model'].str.contains("-1")))].copy()
    if svr_proposed.shape[0]>0:
        proposed_list.append(pd.DataFrame(svr_proposed.iloc[svr_proposed['val_metric'].argmin()]).T)

    mlp_proposed = df[( (df['model'].str.contains("mlp-")) )].copy()
    if mlp_proposed.shape[0]>0:
        proposed_list.append(pd.DataFrame(mlp_proposed.iloc[mlp_proposed['val_metric'].argmin()]).T)

    proposed_list.append(
        df[df['model'].isin(
            ['arima', 'arimamlp', 'arimasvr', 'mlpgs', 'svrgs']
        )]
    )

In [297]:
pd.concat(proposed_list)

,ts,model,MSE,RMSE,MAPE,MAE,theil,ARV,IA,POCID,val_metric,time_testing,time_training
13,Unemployment,mlp-5-0.5,1537.350999,39.197924,5.04158,29.994161,0.323482,0.130086,0.966939,71.041667,30.410345,0.001313,0.258301
0,Unemployment,arima,2978.55064,54.5761,7.012015,42.185424,0.459857,0.230224,0.937808,64.583333,inf,inf,inf
1,Unemployment,arimamlp,1507.623215,38.015974,4.99482,29.138739,0.255764,0.119458,0.968734,68.75,29.110805,0.000266,1.12956
2,Unemployment,arimasvr,1275.651599,35.716265,4.628084,26.542111,0.255639,0.111756,0.972213,70.833333,29.210214,0.001345,0.012543
15,Unemployment,mlpgs,1257.368291,35.450172,4.457729,26.569295,0.446288,0.120187,0.971427,75.625,28.047807,0.000295,0.104141
16,Unemployment,svrgs,1935.252226,43.991502,5.36371,33.372764,0.817258,0.253417,0.948664,77.083333,32.806197,0.001205,0.008954
29,ausbee,mlp-5-0.1,187.663518,13.698986,2.467034,10.466893,0.051816,0.103955,0.971565,95.238095,16.825221,0.001105,0.017484
17,ausbee,arima,185.170053,13.60772,2.445095,10.380154,0.051159,0.102756,0.971924,95.238095,inf,inf,inf
18,ausbee,arimamlp,206.236338,14.355229,2.596583,11.002181,0.056505,0.112594,0.968929,95.238095,17.761218,0.00024,0.071745
19,ausbee,arimasvr,197.724956,14.061471,2.573711,10.889252,0.054385,0.109454,0.970015,95.238095,17.185626,0.000352,0.001207


ts                    woolyrnq
model              svr-10-0.01
MSE              179419.653158
RMSE                423.579571
MAPE                  6.355562
MAE                  303.52592
theil                  0.37125
ARV                   0.551535
IA                    0.858312
POCID                 78.26087
val_metric          322.010763
time_testing          0.004785
time_training         0.109223
Name: 118, dtype: object

In [276]:
df_mean_metrics.reset_index()['ts'].nunique()

14

In [277]:
df_arima.sum(axis=1)

model
arima           0
arimamlp        5
arimasvr        7
mlp-1-0.01     10
mlp-1-0.1      10
mlp-1-0.5       6
mlp-1-1         4
mlp-10-0.01     9
mlp-10-0.1      9
mlp-10-0.5      5
mlp-10-1        5
mlp-2-0.01      9
mlp-2-0.1       8
mlp-2-0.5       5
mlp-2-1         3
mlp-20-0.01     9
mlp-20-0.1      9
mlp-20-0.5      5
mlp-20-1        5
mlp-3-0.01      9
mlp-3-0.1       7
mlp-3-0.5       5
mlp-3-1         6
mlp-40-0.01     9
mlp-40-0.1      7
mlp-40-0.5      5
mlp-40-1        5
mlp-5-0.01      8
mlp-5-0.1       7
mlp-5-0.5       7
mlp-5-1         5
mlpgs           6
svrgs           6
dtype: object

In [122]:
df_mlp.sum(axis=1)

model
arima               8
arimamlp            9
arimasvr            9
lrcmlp-15-0.01      7
lrcmlp-15-0.1       8
lrcmlp-15-0.5       7
lrcmlp-15-1         7
lrcmlp-3-0.01       7
lrcmlp-3-0.1        7
lrcmlp-3-0.5        7
lrcmlp-3-1          5
lrcmlp-30-0.01      7
lrcmlp-30-0.1       8
lrcmlp-30-0.5       5
lrcmlp-30-1         5
lrcmlp-5-0.01       7
lrcmlp-5-0.1        6
lrcmlp-5-0.5        7
lrcmlp-5-1          6
lrcmlp-60-0.01      6
lrcmlp-60-0.1       7
lrcmlp-60-0.5       6
lrcmlp-60-1         5
lrcmlpinv-10-0.9    8
lrcmlpinv-2-0.9     8
lrcmlpinv-20-0.9    7
lrcmlpinv-4-0.9     8
lrcmlpinv-6-0.9     7
lrcmlpinv-8-0.9     7
lrcsvr-15-0.01      7
lrcsvr-15-0.1       7
lrcsvr-15-0.5       3
lrcsvr-15-1         3
lrcsvr-2-0.9        6
lrcsvr-3-0.01       7
lrcsvr-3-0.1        7
lrcsvr-3-0.5        7
lrcsvr-3-1          6
lrcsvr-30-0.01      7
lrcsvr-30-0.1       5
lrcsvr-30-0.5       2
lrcsvr-30-1         2
lrcsvr-4-0.9        7
lrcsvr-5-0.01       7
lrcsvr-5-0.1        6
lrcs

In [123]:
df_svr.sum(axis=1)

model
arima                8
arimamlp            10
arimasvr             9
lrcmlp-15-0.01       7
lrcmlp-15-0.1        8
lrcmlp-15-0.5        7
lrcmlp-15-1          7
lrcmlp-3-0.01        7
lrcmlp-3-0.1         6
lrcmlp-3-0.5         7
lrcmlp-3-1           5
lrcmlp-30-0.01       6
lrcmlp-30-0.1        7
lrcmlp-30-0.5        7
lrcmlp-30-1          5
lrcmlp-5-0.01        7
lrcmlp-5-0.1         7
lrcmlp-5-0.5         7
lrcmlp-5-1           7
lrcmlp-60-0.01       8
lrcmlp-60-0.1        8
lrcmlp-60-0.5        6
lrcmlp-60-1          5
lrcmlpinv-10-0.9     7
lrcmlpinv-2-0.9      8
lrcmlpinv-20-0.9     7
lrcmlpinv-4-0.9      8
lrcmlpinv-6-0.9      7
lrcmlpinv-8-0.9      6
lrcsvr-15-0.01       7
lrcsvr-15-0.1        7
lrcsvr-15-0.5        5
lrcsvr-15-1          5
lrcsvr-2-0.9         6
lrcsvr-3-0.01        7
lrcsvr-3-0.1         7
lrcsvr-3-0.5         6
lrcsvr-3-1           6
lrcsvr-30-0.01       7
lrcsvr-30-0.1        6
lrcsvr-30-0.5        4
lrcsvr-30-1          4
lrcsvr-4-0.9         6
lrcsv